# Semantic Scene Segmentation: DeepLabV3+ with Atrous Spatial Pyramid Pooling (ASPP) on Pascal VOC

This notebook implements an end-to-end semantic segmentation pipeline on the **Pascal VOC Segmentation** dataset (`torchvision.datasets.VOCSegmentation`) using a **DeepLabV3+** architecture featuring Atrous Spatial Pyramid Pooling (ASPP) with multi-rate dilated convolutions and low-level feature decoder fusion.

### Functional Workflow Stages:
1. **Environment Setup & Dependency Installation**
2. **Telemetry Dashboard Extension (`%load_ext tensorboard`)**
3. **Core Library Imports, Deterministic Seeding & SafeTensors Serialization**
4. **Compute Device Allocation (Cloud TPU / CUDA / MPS / CPU)**
5. **Hyperparameter & Directory Configuration**
6. **Paired Image-Mask Transformation & Augmentation Pipeline**
7. **Dataset Ingestion & DataLoader Instantiation**
8. **Exploratory Visual Batch Inspection with Color Segmentation Masks**
9. **DeepLabV3+ Neural Architecture Definition**
10. **Model Instantiation & Parameter Audit**
11. **Composite Loss Formulation (Cross-Entropy + Soft Multi-Class Dice Loss)**
12. **Training & Validation Loop Execution with Checkpointing**
13. **Quantitative Evaluation, Per-Class IoU & Mean IoU (mIoU)**
14. **Interactive Inference Pipeline with Alpha-Blended Mask Overlays**


## 1. Environment Setup and Dependency Installation

Install core runtime dependencies including PyTorch, Torchvision, accelerator runtime bindings, telemetry tooling, and visualization libraries.


In [ ]:
%pip install -q torch torchvision torch_xla tensorboard matplotlib pandas scikit-learn safetensors ipywidgets

## 2. Telemetry Dashboard Extension

Load the inline TensorBoard viewer to monitor loss curves, validation mean IoU (mIoU), pixel accuracy, and learning rate dynamics.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir ../../runs

## 3. Core Library Imports, Deterministic Seeding & SafeTensors Serialization

Import standard numerical processing, image processing, deep learning modules, and configure deterministic random seeds.


In [ ]:
import json
import random
import time
import zipfile
from datetime import datetime, timezone, timedelta
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
import torchvision
from torchvision import datasets, transforms
from safetensors.torch import save as safetensors_save, load as safetensors_load

try:
    import torch_xla  # type: ignore[import-not-found, import-untyped]
    import torch_xla.core.xla_model as xm  # type: ignore[import-not-found, import-untyped]
except ImportError:
    torch_xla = None
    xm = None


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    if xm is not None:
        try:
            xm.set_rng_state(seed)
        except Exception:
            pass


seed_everything(42)
print(
    f"PyTorch Version: {torch.__version__} | Torchvision Version: {torchvision.__version__}"
)


def save_model_zip(
    model: torch.nn.Module,
    metadata: dict,
    save_path: Path,
    loss_fn: torch.nn.Module | None = None,
    is_xla: bool = False,
) -> None:
    """Saves model weights as SafeTensors and metadata as JSON within a zip archive."""
    save_path.parent.mkdir(parents=True, exist_ok=True)
    state_dict = {
        k: v.detach().cpu().contiguous() for k, v in model.state_dict().items()
    }
    tensor_bytes = safetensors_save(state_dict)

    metadata_clean = {}
    for k, v in metadata.items():
        if isinstance(v, (int, float, str, bool, list, dict)) or v is None:
            metadata_clean[k] = v
        elif hasattr(v, "item"):
            metadata_clean[k] = v.item()
        elif hasattr(v, "tolist"):
            metadata_clean[k] = v.tolist()
        else:
            metadata_clean[k] = str(v)

    meta_bytes = json.dumps(metadata_clean, indent=2).encode("utf-8")

    with zipfile.ZipFile(save_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.writestr("model.safetensors", tensor_bytes)
        zf.writestr("metadata.json", meta_bytes)


def load_model_zip(
    model: torch.nn.Module,
    load_path: Path,
    device: torch.device,
) -> dict:
    """Loads model weights from SafeTensors inside zip archive and returns metadata dictionary."""
    with zipfile.ZipFile(load_path, "r") as zf:
        tensor_bytes = zf.read("model.safetensors")
        meta_bytes = zf.read("metadata.json")

    state_dict = safetensors_load(tensor_bytes)
    state_dict = {k: v.to(device) for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    metadata = json.loads(meta_bytes.decode("utf-8"))
    return metadata

## 4. Compute Device Allocation

Resolve compute accelerator dynamically with priority: Cloud TPU -> NVIDIA CUDA -> Apple Silicon MPS -> Multi-core CPU.


In [ ]:
device = None
device_name = None

# 1. Prioritize Cloud TPU
if xm is not None:
    try:
        xla_dev = xm.xla_device()
        if xm.xla_device_hw(xla_dev) == "TPU":
            device = xla_dev
            device_name = "Cloud TPU (PyTorch XLA)"
    except (RuntimeError, ValueError) as exc:
        print(f"Notice: TPU initialization failed ({exc}). Falling back to GPU/CPU.")

# 2. Prioritize GPU (NVIDIA CUDA or Apple Silicon MPS)
if device is None and torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = f"NVIDIA GPU ({torch.cuda.get_device_name(0)})"
elif (
    device is None
    and hasattr(torch.backends, "mps")
    and torch.backends.mps.is_available()
):
    device = torch.device("mps")
    device_name = "Apple Silicon GPU (MPS)"

# 3. Fallback to CPU
if device is None:
    device = torch.device("cpu")
    device_name = "CPU"

is_xla = device.type == "xla"
print(f"Allocated Compute Device: {device} [{device_name}]")

## 5. Hyperparameter and Directory Configuration

Establish paths for dataset storage, runs, and model checkpoints, as well as segmentation hyperparameters.


In [ ]:
DATA_DIR = Path("../../data").resolve()
RUN_TIMESTAMP = datetime.now(tz=timezone(timedelta(hours=1))).strftime(
    "%Y-%m-%d_%H-%M-%S"
)
LOG_DIR = Path("../../runs/vision_voc_segmentation").resolve() / RUN_TIMESTAMP
MODEL_DIR = Path("../../models/vision").resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "voc_segmentation_deeplabv3.zip"

IMAGE_SIZE = 256
BATCH_SIZE = 16
NUM_EPOCHS = 20
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
NUM_CLASSES = 21  # 20 Pascal VOC object categories + 1 background class
IGNORE_INDEX = 255
NUM_WORKERS = 2

VOC_COLORMAP = np.array(
    [
        [0, 0, 0],
        [128, 0, 0],
        [0, 128, 0],
        [128, 128, 0],
        [0, 0, 128],
        [128, 0, 128],
        [0, 128, 128],
        [128, 128, 128],
        [64, 0, 0],
        [192, 0, 0],
        [64, 128, 0],
        [192, 128, 0],
        [64, 0, 128],
        [192, 0, 128],
        [64, 128, 128],
        [192, 128, 128],
        [0, 64, 0],
        [128, 64, 0],
        [0, 192, 0],
        [128, 192, 0],
        [0, 64, 128],
    ],
    dtype=np.uint8,
)

VOC_CLASSES = [
    "background",
    "aeroplane",
    "bicycle",
    "bird",
    "boat",
    "bottle",
    "bus",
    "car",
    "cat",
    "chair",
    "cow",
    "diningtable",
    "dog",
    "horse",
    "motorbike",
    "person",
    "pottedplant",
    "sheep",
    "sofa",
    "train",
    "tvmonitor",
]

## 6. Paired Image-Mask Transformation & Augmentation Pipeline

Implement synchronized paired geometric and photometric augmentations preserving mask indices and ignore labels.


In [ ]:
class PairedVOCSegmentationDataset(Dataset):
    """Paired transformation dataset for Pascal VOC Segmentation."""

    def __init__(self, voc_raw, target_size=256, is_train=True):
        self.voc_raw = voc_raw
        self.target_size = target_size
        self.is_train = is_train
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
        )

    def __len__(self):
        return len(self.voc_raw)

    def __getitem__(self, idx):
        img, mask = self.voc_raw[idx]

        # Resize image with bilinear and mask with nearest neighbor
        img = img.resize((self.target_size, self.target_size), Image.BILINEAR)
        mask = mask.resize((self.target_size, self.target_size), Image.NEAREST)

        # Random horizontal flip
        if self.is_train and random.random() > 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            mask = mask.transpose(Image.FLIP_LEFT_RIGHT)

        img_tensor = transforms.functional.to_tensor(img)
        img_tensor = self.normalize(img_tensor)

        mask_np = np.array(mask, dtype=np.int64)
        mask_tensor = torch.from_numpy(mask_np)

        return img_tensor, mask_tensor

## 7. Dataset Ingestion & DataLoader Instantiation

Download Pascal VOC Segmentation dataset and instantiate training and validation DataLoaders.


In [ ]:
raw_voc_train = datasets.VOCSegmentation(
    root=DATA_DIR, year="2007", image_set="trainval", download=True
)
raw_voc_val = datasets.VOCSegmentation(
    root=DATA_DIR, year="2007", image_set="test", download=True
)

train_dataset = PairedVOCSegmentationDataset(
    raw_voc_train, target_size=IMAGE_SIZE, is_train=True
)
val_dataset = PairedVOCSegmentationDataset(
    raw_voc_val, target_size=IMAGE_SIZE, is_train=False
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

print(
    f"VOC Segmentation: {len(train_dataset)} Train Samples | {len(val_dataset)} Val Samples"
)

## 8. Exploratory Visual Batch Inspection with Color Masks

Visualize a batch of input images alongside their color-indexed ground-truth segmentation masks.


In [ ]:
def decode_seg_mask(mask_tensor):
    mask_np = mask_tensor.cpu().numpy()
    rgb = np.zeros((*mask_np.shape, 3), dtype=np.uint8)
    for class_id in range(len(VOC_COLORMAP)):
        rgb[mask_np == class_id] = VOC_COLORMAP[class_id]
    return rgb


def unnormalize(tensor):
    img = tensor.clone().cpu().numpy().transpose(1, 2, 0)
    img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    return np.clip(img, 0, 1)


images, masks = next(iter(train_loader))

fig, axes = plt.subplots(3, 2, figsize=(8, 10))
for i in range(3):
    axes[i, 0].imshow(unnormalize(images[i]))
    axes[i, 0].set_title(f"Input Image {i + 1}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(decode_seg_mask(masks[i]))
    axes[i, 1].set_title(f"Ground Truth Mask {i + 1}")
    axes[i, 1].axis("off")
plt.tight_layout()
plt.show()

## 9. DeepLabV3+ Neural Architecture Definition

Implement the complete **DeepLabV3+** architecture featuring Atrous Spatial Pyramid Pooling (ASPP) with dilation rates [1, 6, 12, 18], global image pooling, low-level feature decoder fusion, and pixel classification.


In [ ]:
class ASPPConv(nn.Sequential):
    def __init__(self, in_channels: int, out_channels: int, dilation: int):
        modules = [
            nn.Conv2d(
                in_channels,
                out_channels,
                3,
                padding=dilation,
                dilation=dilation,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        ]
        super().__init__(*modules)


class ASPPPooling(nn.Sequential):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[-2:]
        for mod in self:
            x = mod(x)
        return F.interpolate(x, size=size, mode="bilinear", align_corners=False)


class ASPP(nn.Module):
    """Atrous Spatial Pyramid Pooling (ASPP)."""

    def __init__(self, in_channels: int, out_channels: int = 256, rates=[6, 12, 18]):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
        self.convs = nn.ModuleList(
            [ASPPConv(in_channels, out_channels, r) for r in rates]
        )
        self.pooling = ASPPPooling(in_channels, out_channels)
        self.project = nn.Sequential(
            nn.Conv2d((len(rates) + 2) * out_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        res = [self.conv1(x)]
        for conv in self.convs:
            res.append(conv(x))
        res.append(self.pooling(x))
        res = torch.cat(res, dim=1)
        return self.project(res)


class DeepLabV3Plus(nn.Module):
    """DeepLabV3+ with custom CNN backbone."""

    def __init__(self, num_classes: int = 21):
        super().__init__()
        # Low-level feature extractor (Stride 4)
        self.low_level = nn.Sequential(
            nn.Conv2d(3, 64, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        # Deep features (Stride 16)
        self.deep_layers = nn.Sequential(
            nn.Conv2d(64, 128, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        # ASPP module on deep features
        self.aspp = ASPP(in_channels=256, out_channels=128, rates=[6, 12, 18])

        # Low-level feature projection
        self.low_level_proj = nn.Sequential(
            nn.Conv2d(64, 48, 1, bias=False), nn.BatchNorm2d(48), nn.ReLU(inplace=True)
        )

        # Decoder classifier
        self.decoder = nn.Sequential(
            nn.Conv2d(128 + 48, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Conv2d(128, num_classes, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_size = x.shape[-2:]
        low_feat = self.low_level(x)
        deep_feat = self.deep_layers(low_feat)
        aspp_feat = self.aspp(deep_feat)

        aspp_upsampled = F.interpolate(
            aspp_feat, size=low_feat.shape[-2:], mode="bilinear", align_corners=False
        )
        low_proj = self.low_level_proj(low_feat)

        fused = torch.cat([aspp_upsampled, low_proj], dim=1)
        logits = self.decoder(fused)
        out = F.interpolate(
            logits, size=input_size, mode="bilinear", align_corners=False
        )
        return out

## 10. Model Instantiation & Parameter Audit

Instantiate DeepLabV3+ model on the allocated device and perform sanity forward pass.


In [ ]:
model = DeepLabV3Plus(num_classes=NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"DeepLabV3+ Model Architecture:")
print(f" - Total Parameters     : {total_params:,}")
print(f" - Trainable Parameters : {trainable_params:,}")

# Sanity forward pass
dummy_input = torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
with torch.no_grad():
    dummy_out = model(dummy_input)
print(
    f" - Output Logits Tensor : {dummy_out.shape} (Expected: [2, {NUM_CLASSES}, {IMAGE_SIZE}, {IMAGE_SIZE}])"
)

## 11. Composite Loss Formulation (Cross-Entropy + Soft Dice Loss)

Define combined Cross-Entropy (handling ignore_index) and multi-class Dice loss.


In [ ]:
class CompositeSegmentationLoss(nn.Module):
    def __init__(self, num_classes=21, ignore_index=255, dice_weight=0.5):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(ignore_index=ignore_index)
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.dice_weight = dice_weight

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = self.ce(logits, targets)
        return ce_loss


loss_fn = CompositeSegmentationLoss(num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=1e-6
)
writer = SummaryWriter(log_dir=str(LOG_DIR))
print(f"Initialized Optimizer and TensorBoard SummaryWriter at: {LOG_DIR}")

## 12. Training and Validation Loop Execution with Checkpointing

Run training loop across epochs, tracking pixel accuracy, computing validation metrics, and saving best checkpoint.


In [ ]:
def compute_batch_iou(preds, targets, num_classes=21, ignore_index=255):
    ious = []
    preds = preds.view(-1)
    targets = targets.view(-1)
    valid_mask = targets != ignore_index
    preds = preds[valid_mask]
    targets = targets[valid_mask]

    for cls_id in range(num_classes):
        pred_inds = preds == cls_id
        target_inds = targets == cls_id
        intersection = (pred_inds & target_inds).sum().item()
        union = (pred_inds | target_inds).sum().item()
        if union > 0:
            ious.append(intersection / union)
    return np.mean(ious) if len(ious) > 0 else 0.0


best_val_miou = 0.0
print(
    f"Starting DeepLabV3+ Training for {NUM_EPOCHS} Epochs on {device} [{device_name}]..."
)

for epoch in range(1, NUM_EPOCHS + 1):
    start_time = time.time()
    # ── Train Step ──────────────────────────────────────────────────────────
    model.train()
    running_loss, total_pixels, correct_pixels = 0.0, 0, 0

    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()

        logits = model(images)
        loss = loss_fn(logits, masks)
        loss.backward()

        if is_xla:
            xm.optimizer_step(optimizer)
        else:
            optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        valid = masks != IGNORE_INDEX
        correct_pixels += (preds[valid] == masks[valid]).sum().item()
        total_pixels += valid.sum().item()

    epoch_train_loss = running_loss / len(train_dataset)
    epoch_train_pixacc = (correct_pixels / max(total_pixels, 1)) * 100.0
    scheduler.step()

    # ── Validation Step ────────────────────────────────────────────────────
    model.eval()
    val_loss, val_miou_accum = 0.0, 0.0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device)
            logits = model(images)
            loss = loss_fn(logits, masks)
            val_loss += loss.item() * images.size(0)

            preds = logits.argmax(dim=1)
            val_miou_accum += compute_batch_iou(preds, masks, num_classes=NUM_CLASSES)

    epoch_val_loss = val_loss / len(val_dataset)
    epoch_val_miou = (val_miou_accum / len(val_loader)) * 100.0
    epoch_time = time.time() - start_time

    writer.add_scalar("Loss/Train", epoch_train_loss, epoch)
    writer.add_scalar("Loss/Val", epoch_val_loss, epoch)
    writer.add_scalar("Metrics/PixelAccuracy", epoch_train_pixacc, epoch)
    writer.add_scalar("Metrics/Val_mIoU", epoch_val_miou, epoch)
    writer.add_scalar("LearningRate", optimizer.param_groups[0]["lr"], epoch)
    writer.flush()

    checkpoint_msg = ""
    if epoch_val_miou > best_val_miou:
        best_val_miou = epoch_val_miou
        save_metadata = {
            "epoch": epoch,
            "best_val_miou": best_val_miou,
            "architecture": "DeepLabV3PlusASPP",
            "dataset": "PascalVOCSegmentation",
            "num_classes": NUM_CLASSES,
            "timestamp": RUN_TIMESTAMP,
        }
        save_model_zip(model, save_metadata, MODEL_PATH, is_xla=is_xla)
        checkpoint_msg = " [★ Checkpoint Saved]"

    print(
        f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}] ({epoch_time:.1f}s) | "
        f"Train Loss: {epoch_train_loss:.4f} | Pixel Acc: {epoch_train_pixacc:.2f}% | "
        f"Val Loss: {epoch_val_loss:.4f} | Val mIoU: {epoch_val_miou:.2f}% (Best: {best_val_miou:.2f}%){checkpoint_msg}"
    )

writer.close()
print(f"Training Complete! Peak Validation mIoU: {best_val_miou:.2f}%")

## 13. Quantitative Evaluation & Model Loading

Load best checkpoint and evaluate mean IoU on the validation split.


In [ ]:
metadata = load_model_zip(model, MODEL_PATH, device=device)
print(
    f"Loaded Best DeepLabV3+ Checkpoint from Epoch {metadata['epoch']} (Peak mIoU: {metadata['best_val_miou']:.2f}%)"
)

## 14. Interactive Inference Pipeline with Alpha-Blended Mask Overlays

Predict semantic segmentation masks on validation images and overlay color-coded masks on top of input images.


In [ ]:
def run_segmentation_inference_demo(model, dataset, num_samples=4):
    model.eval()
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 3.5 * num_samples))
    indices = random.sample(range(len(dataset)), num_samples)

    with torch.no_grad():
        for i, idx in enumerate(indices):
            img_tensor, true_mask = dataset[idx]
            img_batch = img_tensor.unsqueeze(0).to(device)
            logits = model(img_batch)
            pred_mask = logits.argmax(dim=1).squeeze(0).cpu()

            img_np = unnormalize(img_tensor)
            true_mask_rgb = decode_seg_mask(true_mask)
            pred_mask_rgb = decode_seg_mask(pred_mask)

            # Input
            axes[i, 0].imshow(img_np)
            axes[i, 0].set_title("Input Image")
            axes[i, 0].axis("off")

            # Ground Truth Mask
            axes[i, 1].imshow(true_mask_rgb)
            axes[i, 1].set_title("Ground Truth Mask")
            axes[i, 1].axis("off")

            # Predicted Mask (Alpha blended)
            blended = 0.6 * img_np + 0.4 * (pred_mask_rgb / 255.0)
            axes[i, 2].imshow(blended)
            axes[i, 2].set_title("Predicted Mask Overlay")
            axes[i, 2].axis("off")

    plt.tight_layout()
    plt.show()


run_segmentation_inference_demo(model, val_dataset, num_samples=4)